# Demo M1/M2/M3 — app no Colab

Este notebook sobe o **app web** (`app.py`) com os três braços do estudo:

- **M1** — pipeline (Markov + regressões), instantâneo.
- **M2** — o M1 com o texto de regras refinado pelo LLM.
- **M3** — LLM pura (`gemma3:4b`).

O app roda **inteiro no Colab**, inclusive o Ollama. No fim, `share=True`
devolve um **link público temporário**: quem receber abre no navegador e usa
sem instalar nada.

> **Use um runtime com GPU** (`Ambiente de execução → Alterar tipo de
> ambiente → T4 GPU`). Na CPU o `gemma3:4b` funciona, mas leva minutos por
> carta; o M1 continua instantâneo.
>
> O link vive enquanto a sessão estiver aberta (o Colab encerra sozinho
> depois de algumas horas). Para uma demo em aula, suba o app alguns minutos
> antes e deixe a aba aberta.

In [ ]:
%cd /content

# Clona o repositório (ou atualiza, se a pasta já existir nesta sessão).
!git clone -q https://github.com/funkymonks21/MTGCardCreator-IFRS.git 2>/dev/null || (cd MTGCardCreator-IFRS && git pull -q)

%cd MTGCardCreator-IFRS
!pip install -q -r requirements-app.txt
print("Dependências instaladas.")

## 1. Ollama + `gemma3:4b`

O M2/M3 falam com o Ollama em `http://localhost:11434` — o mesmo backend e o
mesmo modelo usados no estudo. Baixar o modelo leva alguns minutos (~3,4 GB)
na primeira execução; nas seguintes o cache da sessão ainda não persiste, mas
o download é rápido.

In [ ]:
# Instala o Ollama (script oficial) e sobe o serviço em segundo plano.
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    print("O Ollama não subiu. Veja o log em /content/ollama.log")

!ollama pull gemma3:4b

In [ ]:
import sys
sys.path.insert(0, "src")
from pipeline import ollama_disponivel

print("GPU:")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()
print("Ollama:", "disponível" if ollama_disponivel() else "INDISPONÍVEL")
!ollama list

## 2. Recursos do NLTK

A inferência automática de tipo pelo nome usa POS tagging e WordNet. São
downloads pequenos, feitos uma vez por sessão.

In [ ]:
import nltk

for pacote in [
    "wordnet", "omw-1.4", "punkt_tab",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
]:
    nltk.download(pacote, quiet=True)

print("NLTK pronto.")

## 3. Subir o app

`share=True` cria o link público (a URL aparece na saída abaixo, no formato
`https://....gradio.live`). Abra em outra aba para conferir antes de
compartilhar.

A primeira geração do M1 carrega os modelos (~10 s); depois é instantânea.
M2/M3 levam alguns segundos por carta.

In [ ]:
from app import criar_app

demo = criar_app()
demo.queue(default_concurrency_limit=2).launch(
    share=True,
    show_error=True,
)

## Dicas para a aula

- **Comparar os três**: aba *Comparar os três*. Com a mesma semente, M1 e M2
  têm estrutura idêntica — a diferença visível é o texto.
- **Arte**: ligada por padrão; sai da API gratuita (Pollinations, ~1 imagem a
  cada 45 s). Se estourar o limite, a carta usa um fundo procedural e avisa.
- **Ollama caiu?** Rode a célula do Ollama de novo. O M1 continua funcionando
  mesmo sem ele.
- **Link expirou?** Enquanto a sessão viver, rodar a última célula gera um
  link novo.